In [39]:
# If needed:
# !pip install ultralytics torchmetrics opencv-python

import sys
from pathlib import Path

import xml.etree.ElementTree as ET

import cv2
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from tqdm import tqdm

from ultralytics import YOLO
from torchmetrics.detection.mean_ap import MeanAveragePrecision

In [40]:
# Configuration

from pathlib import Path
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ROOT = Path.cwd().parent  # since you're in /notebooks
DATASET_NAME = "MVI_20011"

DATA_DIR = ROOT / "data"

XML_PATH = DATA_DIR / f"{DATASET_NAME}.xml"
IMG_DIR = DATA_DIR / DATASET_NAME
YOLO_WEIGHTS = ROOT / "model" / "best.pt"
model = YOLO(YOLO_WEIGHTS)

print("Model exists:", YOLO_WEIGHTS.exists())

print("XML:", XML_PATH, XML_PATH.exists())
print("IMG:", IMG_DIR, IMG_DIR.exists())
print("MODEL:", YOLO_WEIGHTS, YOLO_WEIGHTS.exists())

BATCH_SIZE = 8
NUM_WORKERS = 2
CONF_THRESH = 0.30
IOU_MATCH_THRESH = 0.50

# UA-DETRAC / YOLO class mapping
CLASS_TO_ID = {
    "car": 0,
    "bus": 1,
    "van": 2,
    "others": 3
}

ID_TO_CLASS = {v: k for k, v in CLASS_TO_ID.items()}

# YOLO already predicts 0..3, so no offset needed
YOLO_LABEL_OFFSET = 0



Model exists: True
XML: c:\Users\ameli\OneDrive\Desktop\miniproject2\data\MVI_20011.xml True
IMG: c:\Users\ameli\OneDrive\Desktop\miniproject2\data\MVI_20011 True
MODEL: c:\Users\ameli\OneDrive\Desktop\miniproject2\model\best.pt True


In [41]:
from pathlib import Path

ROOT = Path.cwd().parent

DATA_YAML = ROOT / "data.yaml"

DATA_YAML.write_text("""path: data

train: MVI_20012
val: MVI_20011

names:
  0: car
  1: van
  2: bus
  3: others
""")

print(DATA_YAML.exists(), DATA_YAML)

True c:\Users\ameli\OneDrive\Desktop\miniproject2\data.yaml


In [42]:
# Parse UA-DETRAC XML ground truth

def parse_detrac_xml(xml_path: str):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    annotations = {}
    for frame in root.findall(".//frame"):
        fid = int(frame.get("num"))
        target_list = frame.find("target_list")

        boxes, labels = [], []
        if target_list is not None:
            for target in target_list.findall("target"):
                attr = target.find("attribute")
                box = target.find("box")
                if attr is None or box is None:
                    continue

                vtype = attr.get("vehicle_type", "others").lower()
                if vtype not in CLASS_TO_ID:
                    vtype = "others"

                left = float(box.get("left", 0))
                top = float(box.get("top", 0))
                width = float(box.get("width", 0))
                height = float(box.get("height", 0))

                x1, y1 = left, top
                x2, y2 = left + width, top + height
                if x2 <= x1 or y2 <= y1:
                    continue

                boxes.append([x1, y1, x2, y2])
                labels.append(CLASS_TO_ID[vtype])

        if boxes:
            annotations[fid] = {
                "boxes": torch.tensor(boxes, dtype=torch.float32),
                "labels": torch.tensor(labels, dtype=torch.int64),
            }

    return annotations

In [43]:
# =========================
# 4) Dataset
# =========================
class DetracEvalDataset(Dataset):
    def __init__(self, img_dir, annotations):
        self.img_dir = Path(img_dir)
        self.annotations = annotations
        self.frame_ids = sorted(list(annotations.keys()))
        self.to_tensor = T.ToTensor()

    def __len__(self):
        return len(self.frame_ids)

    def __getitem__(self, idx):
        fid = self.frame_ids[idx]
        img_path = self.img_dir / f"img{fid:05d}.jpg"

        img_bgr = cv2.imread(str(img_path))
        if img_bgr is None:
            raise FileNotFoundError(f"Missing image: {img_path}")

        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_tensor = self.to_tensor(img_rgb)

        target = {
            "boxes": self.annotations[fid]["boxes"].clone(),
            "labels": self.annotations[fid]["labels"].clone(),
            "image_id": torch.tensor([fid], dtype=torch.int64),
            "img_path": str(img_path),  # useful for YOLO batch inference
        }
        return img_tensor, target

def collate_fn(batch):
    images, targets = zip(*batch)
    return list(images), list(targets)

In [44]:
# =========================
# 5) IoU + P/R/F1 helpers

def box_iou_xyxy(boxes1, boxes2):
    if boxes1.numel() == 0 or boxes2.numel() == 0:
        return torch.zeros((boxes1.shape[0], boxes2.shape[0]), dtype=torch.float32)

    area1 = (boxes1[:, 2] - boxes1[:, 0]).clamp(min=0) * (boxes1[:, 3] - boxes1[:, 1]).clamp(min=0)
    area2 = (boxes2[:, 2] - boxes2[:, 0]).clamp(min=0) * (boxes2[:, 3] - boxes2[:, 1]).clamp(min=0)

    lt = torch.max(boxes1[:, None, :2], boxes2[:, :2])
    rb = torch.min(boxes1[:, None, 2:], boxes2[:, 2:])
    wh = (rb - lt).clamp(min=0)
    inter = wh[..., 0] * wh[..., 1]

    union = area1[:, None] + area2 - inter
    return inter / union.clamp(min=1e-6)

@torch.no_grad()
def compute_precision_recall_f1(preds, gts, iou_thresh=0.5, score_thresh=0.3):
    TP, FP, FN = 0, 0, 0

    for pred, gt in zip(preds, gts):
        p_boxes = pred["boxes"]
        p_scores = pred["scores"]
        p_labels = pred["labels"]

        g_boxes = gt["boxes"]
        g_labels = gt["labels"]

        keep = p_scores >= score_thresh
        p_boxes, p_scores, p_labels = p_boxes[keep], p_scores[keep], p_labels[keep]

        classes = torch.unique(torch.cat([p_labels, g_labels], dim=0)) if (len(p_labels) or len(g_labels)) else torch.tensor([])
        for c in classes.tolist():
            pb = p_boxes[p_labels == c]
            ps = p_scores[p_labels == c]
            gb = g_boxes[g_labels == c]

            if len(pb) == 0 and len(gb) == 0:
                continue
            if len(pb) == 0:
                FN += len(gb)
                continue
            if len(gb) == 0:
                FP += len(pb)
                continue

            order = torch.argsort(ps, descending=True)
            pb = pb[order]
            ious = box_iou_xyxy(pb, gb)

            matched_gt = torch.zeros(len(gb), dtype=torch.bool)
            for i in range(len(pb)):
                best_iou, best_j = torch.max(ious[i], dim=0)
                if best_iou >= iou_thresh and not matched_gt[best_j]:
                    TP += 1
                    matched_gt[best_j] = True
                else:
                    FP += 1

            FN += (~matched_gt).sum().item()

    precision = TP / (TP + FP + 1e-9)
    recall = TP / (TP + FN + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)

    return {"TP": TP, "FP": FP, "FN": FN, "precision": precision, "recall": recall, "f1": f1}

In [45]:
# YOLO specific evaluation loop

@torch.no_grad()
def evaluate_yolo(model, dataloader, conf_thresh=0.3, iou_match_thresh=0.5, yolo_label_offset=1):
    metric_map = MeanAveragePrecision(box_format="xyxy", iou_type="bbox", class_metrics=True)

    all_preds = []
    all_gts = []

    for _, targets in tqdm(dataloader, desc="Evaluating YOLO"):
        # YOLO can infer from file paths directly
        img_paths = [t["img_path"] for t in targets]
        results = model.predict(
            source=img_paths,
            conf=conf_thresh,
            iou=0.7,            # NMS IoU, separate from eval IoU
            verbose=False,
            device=0 if DEVICE == "cuda" else "cpu"
        )

        preds_batch = []
        gts_batch = []

        for r, tgt in zip(results, targets):
            boxes = r.boxes.xyxy.cpu() if r.boxes is not None else torch.empty((0, 4), dtype=torch.float32)
            scores = r.boxes.conf.cpu() if r.boxes is not None else torch.empty((0,), dtype=torch.float32)
            labels = r.boxes.cls.cpu().long() + yolo_label_offset if r.boxes is not None else torch.empty((0,), dtype=torch.int64)

            pred = {"boxes": boxes, "scores": scores, "labels": labels}
            gt = {"boxes": tgt["boxes"].cpu(), "labels": tgt["labels"].cpu()}

            preds_batch.append(pred)
            gts_batch.append(gt)

        metric_map.update(preds_batch, gts_batch)
        all_preds.extend(preds_batch)
        all_gts.extend(gts_batch)

    map_result = metric_map.compute()
    prf1_result = compute_precision_recall_f1(
        all_preds, all_gts,
        iou_thresh=iou_match_thresh,
        score_thresh=conf_thresh
    )
    return map_result, prf1_result

In [46]:
# Run Evaluation

annotations = parse_detrac_xml(XML_PATH)
dataset = DetracEvalDataset(IMG_DIR, annotations)
loader = DataLoader(
    dataset,
    batch_size= 4,
    shuffle=False,
    num_workers= 0 ,
    collate_fn=collate_fn
)

model = YOLO(YOLO_WEIGHTS)

map_result, prf1_result = evaluate_yolo(
    model,
    loader,
    conf_thresh=CONF_THRESH,
    iou_match_thresh=0.50,
    yolo_label_offset=YOLO_LABEL_OFFSET
)

print("===== Precision / Recall / F1 =====")
print(f"Precision: {prf1_result['precision']:.4f}")
print(f"Recall:    {prf1_result['recall']:.4f}")
print(f"F1-score:  {prf1_result['f1']:.4f}")
print(f"TP/FP/FN:  {prf1_result['TP']}/{prf1_result['FP']}/{prf1_result['FN']}")

print("\n===== mAP =====")
print(f"mAP@[0.50:0.95]: {map_result['map'].item():.4f}")
print(f"mAP@0.50:        {map_result['map_50'].item():.4f}")
print(f"mAP@0.75:        {map_result['map_75'].item():.4f}")
print(f"mAR@100:         {map_result['mar_100'].item():.4f}")

if map_result.get("map_per_class", None) is not None:
    print("\nClass-wise AP:")
    classes = map_result["classes"].cpu().tolist()
    aps = map_result["map_per_class"].cpu().tolist()
    for cid, ap in zip(classes, aps):
        print(f"{ID_TO_CLASS.get(int(cid), f'class_{int(cid)}')}: {ap:.4f}")

Evaluating YOLO: 100%|██████████| 166/166 [00:20<00:00,  8.01it/s]


===== Precision / Recall / F1 =====
Precision: 0.0000
Recall:    0.0000
F1-score:  0.0000
TP/FP/FN:  0/121/7655

===== mAP =====
mAP@[0.50:0.95]: 0.0000
mAP@0.50:        0.0000
mAP@0.75:        0.0000
mAR@100:         0.0000

Class-wise AP:
car: 0.0000
bus: 0.0000
van: 0.0000
others: 0.0000


In [47]:
# 1 image visual test
sample_img = list(IMG_DIR.glob("*.jpg"))[0]
print(sample_img)

results = model.predict(
    source=str(sample_img),
    conf=0.05,
    save=True
)

print(results[0].boxes)

c:\Users\ameli\OneDrive\Desktop\miniproject2\data\MVI_20011\img00001.jpg
Results saved to C:\Users\ameli\OneDrive\Desktop\miniproject2\notebooks\runs\detect\predict-2
ultralytics.engine.results.Boxes object with attributes:

cls: tensor([1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.])
conf: tensor([0.2296, 0.0988, 0.0897, 0.0806, 0.0788, 0.0766, 0.0741, 0.0663, 0.0632, 0.0605, 0.0538, 0.0521])
data: tensor([[8.4746e+02, 1.6136e+02, 9.6000e+02, 3.7343e+02, 2.2956e-01, 1.0000e+00],
        [5.9654e+02, 7.7508e+01, 6.3385e+02, 1.1279e+02, 9.8802e-02, 0.0000e+00],
        [6.0156e+02, 8.9789e+01, 6.4085e+02, 1.2770e+02, 8.9664e-02, 0.0000e+00],
        [5.8649e+02, 6.0144e+01, 6.1763e+02, 9.1836e+01, 8.0572e-02, 0.0000e+00],
        [6.0524e+02, 9.3968e+01, 6.5093e+02, 1.3416e+02, 7.8752e-02, 0.0000e+00],
        [3.8337e+02, 1.3796e+02, 5.1762e+02, 3.3244e+02, 7.6570e-02, 1.0000e+00],
        [2.9903e+02, 1.6888e+02, 3.4689e+02, 2.1475e+02, 7.4070e-02, 0.0000e+00],
        [5.9063e+02, 7

In [49]:
DATASET_NAME = "MVI_20012"

XML_PATH = DATA_DIR / f"{DATASET_NAME}.xml"
IMG_DIR = DATA_DIR / DATASET_NAME

annotations = parse_detrac_xml(XML_PATH)
dataset = DetracEvalDataset(IMG_DIR, annotations)

loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn
)

map_result, prf1_result = evaluate_yolo(
    model,
    loader,
    conf_thresh=0.05,
    iou_match_thresh=0.25,
    yolo_label_offset=0
)

print(prf1_result)
print(map_result["map_50"])

Evaluating YOLO: 100%|██████████| 234/234 [00:50<00:00,  4.66it/s]


{'TP': 1025, 'FP': 9785, 'FN': 7583, 'precision': 0.09481961147085154, 'recall': 0.11907527881039508, 'f1': 0.10557214905838098}
tensor(0.0038)
